# Day 5 homework — evaluation (project)

Same **patterns** as `course/Day_05_Evaluation.ipynb`, but:

- **Docs agent** with **`doc_search`** over **Pydantic** chunks (not the DE FAQ agent).
- Optional: also log **machine-learning** FAQ runs for comparison.

Uses **`./logs`** under the project folder — keep out of git.


## Setup

```bash
cd project
uv add pandas tqdm requests python-frontmatter minsearch openai python-dotenv pydantic-ai
```

Ensure `OPENAI_API_KEY` is set (e.g. `ai_hero/.env`).


In [ ]:
import asyncio
import json
import os
import random
import secrets
import threading
from datetime import datetime
from pathlib import Path
import io
import zipfile

import pandas as pd
import requests
import frontmatter
from tqdm.auto import tqdm
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.messages import ModelMessagesTypeAdapter
from dotenv import load_dotenv
from minsearch import Index

_p = Path.cwd()
for _ in range(10):
    if (_p / ".env").is_file():
        load_dotenv(_p / ".env")
        break
    if _p.parent == _p:
        break
    _p = _p.parent
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY")

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)


def run_agent_sync(agent, user_prompt: str):
    out = {}
    def _go():
        out["r"] = asyncio.run(agent.run(user_prompt))
    t = threading.Thread(target=_go)
    t.start()
    t.join()
    return out["r"]


In [ ]:
def read_repo_data(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    r = requests.get(url)
    r.raise_for_status()
    out = []
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    for fi in zf.infolist():
        fn = fi.filename
        fl = fn.lower()
        if not (fl.endswith(".md") or fl.endswith(".mdx")):
            continue
        try:
            with zf.open(fi) as f:
                c = f.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(c)
                d = post.to_dict()
                d["filename"] = fn
                out.append(d)
        except Exception as e:
            print("skip", fn, e)
    zf.close()
    return out


def sliding_window(seq, size, step):
    result = []
    for i in range(0, len(seq), step):
        chunk = seq[i : i + size]
        result.append({"start": i, "chunk": chunk})
        if i + size >= len(seq):
            break
    return result


pydantic_docs = read_repo_data("pydantic", "pydantic")
chunks = []
for doc in pydantic_docs:
    dc = doc.copy()
    body = dc.pop("content", "") or ""
    for ch in sliding_window(body, 2000, 1000):
        ch.update(dc)
        chunks.append(ch)

MAX_CHUNKS = 400
_rows = chunks if MAX_CHUNKS is None else chunks[:MAX_CHUNKS]
doc_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[],
)
doc_index.fit(_rows)


def doc_search(query: str):
    return doc_index.search(query, num_results=5)


instructions = """
You answer questions about the Pydantic library using doc_search.
Cite source filenames from search results when possible.
""".strip()

doc_agent = Agent(
    "openai:gpt-4o-mini",
    name="pydantic_doc_agent",
    instructions=instructions,
    tools=[doc_search],
)
len(_rows)


In [ ]:
def _instructions_text(agent) -> str:
    parts = getattr(agent, "_instructions", None) or []
    return "\n\n".join(str(p) for p in parts)


def log_entry(agent, messages, source="user"):
    fts = getattr(agent, "_function_toolset", None)
    tool_names = list(fts.tools.keys()) if fts is not None else []
    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)
    return {
        "agent_name": agent.name,
        "system_prompt": _instructions_text(agent),
        "model": str(getattr(agent, "_model", "")),
        "tools": tool_names,
        "messages": dict_messages,
        "source": source,
    }


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source="user"):
    entry = log_entry(agent, messages, source)
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)
    fp = LOG_DIR / f"{agent.name}_{ts_str}_{rand_hex}.json"
    with fp.open("w", encoding="utf-8") as f:
        json.dump(entry, f, indent=2, default=serializer)
    return fp


In [ ]:
r = run_agent_sync(doc_agent, "How do I use Field with a default factory?")
print(r.output)
log_interaction_to_file(doc_agent, r.new_messages(), source="user")


In [ ]:
evaluation_prompt = """
Evaluate an agent answer (<ANSWER>) for question (<QUESTION>) using <INSTRUCTIONS> and <LOG>.

Checklist (true/false + justification):
- instructions_follow, instructions_avoid, answer_relevant, answer_clear
- answer_citations, completeness, tool_call_search
""".strip()


class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool


class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


eval_agent = Agent(
    "openai:gpt-4o-mini",
    name="eval_agent",
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist,
)

fmt = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()


def load_log_file(path):
    path = Path(path)
    d = json.loads(path.read_text(encoding="utf-8"))
    d["log_file"] = str(path)
    return d


def extract_qa(messages):
    q = a = None
    for m in messages:
        if m.get("kind") == "request":
            for p in m.get("parts", []):
                if p.get("part_kind") == "user-prompt":
                    c = p.get("content")
                    q = c if isinstance(c, str) else str(c)
    for m in reversed(messages):
        if m.get("kind") == "response":
            for p in m.get("parts", []):
                if p.get("part_kind") == "text":
                    c = p.get("content")
                    a = c if isinstance(c, str) else str(c)
                    break
        if a:
            break
    return q, a


def evaluate_one(log_record):
    msgs = log_record["messages"]
    q, a = extract_qa(msgs)
    up = fmt.format(
        instructions=log_record["system_prompt"],
        question=q,
        answer=a,
        log=json.dumps(msgs),
    )
    return run_agent_sync(eval_agent, up).output


In [ ]:
# Evaluate latest log for this agent
logs = sorted(LOG_DIR.glob("pydantic_doc_agent_*.json"))
if logs:
    ev = evaluate_one(load_log_file(logs[-1]))
    print(ev.summary)
    for c in ev.checklist:
        print(c)
else:
    print("No logs yet.")


## Batch evaluation & Pandas

After you have **several** logs (including `source='ai-generated'`), aggregate pass rates per checklist item.

In [ ]:
eval_set = []
for log_file in LOG_DIR.glob("pydantic_doc_agent_*.json"):
    rec = load_log_file(log_file)
    if rec.get("source") == "ai-generated":
        eval_set.append(rec)

eval_results = []
for rec in tqdm(eval_set):
    eval_results.append((rec, evaluate_one(rec)))

rows = []
for rec, ev in eval_results:
    q, a = extract_qa(rec["messages"])
    row = {"file": Path(rec["log_file"]).name, "question": q, "answer": (a or "")[:500]}
    for c in ev.checklist:
        row[c.check_name] = c.check_pass
    rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    print(df.head())
    print(df.mean(numeric_only=True))
else:
    print("No ai-generated logs yet — run the question generator cell below first.")

## AI-generated questions

Sample chunks from the index → LLM writes natural questions → run **`doc_agent`** → log with `source='ai-generated'`. Repeat until you have **≥10** logs for homework.

In [ ]:
qg_prompt = """
You help create test questions for an assistant that answers about the Pydantic Python library.

From each chunk of documentation text, output one realistic question a developer might ask.
Questions should vary: API usage, validation, serialization, performance, migration tips.
""".strip()


class QuestionsList(BaseModel):
    questions: list[str]


question_generator = Agent(
    "openai:gpt-4o-mini",
    name="question_generator",
    instructions=qg_prompt,
    output_type=QuestionsList,
)

sample_rows = random.sample(_rows, min(10, len(_rows)))
snippet_docs = [r.get("chunk", "")[:1500] for r in sample_rows]
qg = run_agent_sync(question_generator, json.dumps(snippet_docs))
questions = qg.output.questions

for q in tqdm(questions):
    print(q)
    res = run_agent_sync(doc_agent, q)
    print(res.output[:400], "...\n")
    log_interaction_to_file(doc_agent, res.new_messages(), source="ai-generated")

## Homework

- Collect **≥10** interaction logs (mix of manual and AI-generated is fine).
- Run the judge on your logs; inspect failures and iterate on **instructions**.
- Duplicate `doc_agent` with a second `instructions` string (e.g. stricter citation rules) and re-run the **same** questions; compare `df.mean(numeric_only=True)` between runs.
- Optional: use a different judge model string in `eval_agent` (e.g. `openai:gpt-4o`) for a second opinion.

[Course](https://alexeygrigorev.com/aihero/)
